Missing or low-quality alt text leaves screen reader users without the information sighted readers get for free, _and_ hurts SEO along the way. Reviewing alt text manually across an entire doc site can be an arduous task, and it's subject to our own sighted biases — how we describe an image may not be useful to someone who can't see it. A vision-powered LLM is better suited to the job.

> This notebook builds on the [Docs Checker](https://inference-docs.cerebras.ai/cookbook/agents/build-a-docs-checker) cookbook. It replaces the `crawl()` function and data models with versions that also capture images, and uses Gemma 4 31B in place of GPT OSS 120B. You don't need to run the original notebook first.

## How it works
 
1. **Crawl + capture** the docs site with Browserbase. While Playwright already has each page loaded for link checking, also screenshot every `<img>` and `<svg>` element and extract the nearby heading and paragraph.
2. **Analyze** each page with Stagehand + Cerebras for broken links, grammar issues, and outdated content — same as the [Docs Checker](https://inference-docs.cerebras.ai/cookbook/agents/build-a-docs-checker).
3. **Scan + caption** the source repo to find every image reference that needs alt text, match it to the captured screenshot, and caption it with Gemma 4 31B using the surrounding page context.
4. **Write back** the suggested alt text into the source files, flag low-confidence suggestions inline, and review the diff before committing.

## Prerequisites

- **Cerebras API key** — [Get one here](https://cloud.cerebras.ai)
- **Browserbase API key** — [Sign up free here](https://www.browserbase.com/)
- **Python 3.10+**

> **Note:** The scanner and write-back step assume your documentation source files are **MDX**. If your source uses plain markdown (`.md`) files, change the glob in `scan_docs` from `*.mdx` to `*.md`.

# Step 1: Environment Setup + API Keys

Install dependencies and configure Cerebras, Browserbase, and Stagehand credentials:

In [ ]:
!pip install playwright browserbase stagehand instructor openai pydantic langchain-community GitPython cerebras-cloud-sdk
!python3 -m playwright install chromium

In [ ]:
import os, asyncio, base64, re, shutil
from collections import deque, Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional, List
from urllib.parse import urlparse

import httpx
from pydantic import BaseModel, Field
from playwright.async_api import async_playwright
from IPython.display import display, clear_output, HTML
import instructor
from openai import AsyncOpenAI

Set your Cerebras and Browserbase credentials. You can also set these as environment variables:

In [ ]:
CEREBRAS_API_KEY = os.getenv("CEREBRAS_API_KEY", "YOUR_CEREBRAS_API_KEY")
BROWSERBASE_API_KEY = os.getenv("BROWSERBASE_API_KEY", "YOUR_BROWSERBASE_API_KEY")
CEREBRAS_MODEL = "gemma-4-31b"
STAGEHAND_MODEL = f"cerebras/{CEREBRAS_MODEL}"

print(f"✓ Configuration loaded")
print(f"  Model: {CEREBRAS_MODEL}")

# Step 2: Extended Data Models

Next, extend the [existing data models](https://inference-docs.cerebras.aihttps://inference-docs.cerebras.ai/cookbook/agents/build-a-docs-checker#step-2-define-data-models) with a new `ImageCapture` pydantic model, and edit `PageResult` to accept images.

In [ ]:
class Issue(BaseModel):
    category: str
    description: str
    page_url: str
    location: Optional[str] = None
    observed: Optional[str] = None
    expected: Optional[str] = None
    evidence: List[str] = Field(default_factory=list)

class LinkResult(BaseModel):
    url: str
    ok: bool
    status: Optional[int] = None
    error: Optional[str] = None

@dataclass
class ImageCapture:
    """A single screenshot captured from a rendered page, plus its context."""
    src: str | None        # None for inline <svg> with no src attribute
    tag: str               # "img" | "svg"
    png_bytes: bytes
    nearby_heading: str
    nearby_paragraph: str
    page_url: str

class PageResult(BaseModel):
    url: str
    issues: List[Issue] = Field(default_factory=list)
    links: List[LinkResult] = Field(default_factory=list)
    images: List[dict] = Field(default_factory=list)  # serialized ImageCaptures

print("✓ Data models defined")

# Step 3: Crawl + Capture

Modify the `crawl()` function to screenshot every `<img>` and `<svg>` element on the page and extract its surrounding section context. The change is in the inner page loop: after collecting links, Playwright will take screenshots. This way, one page load covers both the link check and the image capture.

In [ ]:
from browserbase import Browserbase

MIN_CAPTURE_DIMENSION = 200  # px — enlarge small icons so they're legible for the model

async def _nearest_section_context(element_handle) -> tuple[str, str]:
    """Returns (heading, paragraph) scoped to the element's section.
    Walks backward to the nearest preceding heading, then finds the nearest
    paragraph between that heading and the element — so unrelated text from
    earlier sections doesn't bleed in."""
    return await element_handle.evaluate("""(el) => {
        const isHeading = (n) => /^H[1-6]$/.test(n.tagName);
        const all = Array.from(document.querySelectorAll('h1,h2,h3,h4,h5,h6,p,img,svg'));
        const idx = all.indexOf(el);
        const before = all.slice(0, idx);
        // First pass: walk backward to find the nearest preceding heading
        let headingIdx = -1;
        for (let i = before.length - 1; i >= 0; i--) {
            if (isHeading(before[i])) { headingIdx = i; break; }
        }
        const heading = headingIdx >= 0 ? before[headingIdx].textContent.trim() : '';

        // Second pass: find the nearest paragraph between that heading and this element
        let paragraph = '';
        for (let i = before.length - 1; i > headingIdx; i--) {
            if (before[i].tagName === 'P') { paragraph = before[i].textContent.trim(); break; }
        }
        return [heading, paragraph];
    }""")

async def _capture_page_images(page, page_url: str) -> list[dict]:
    """Screenshot every <img> and <svg> on an already-loaded page.
    Called from inside crawl() while the page is already open."""
    captures = []
    for tag in ("img", "svg"):
        for el in await page.query_selector_all(tag):
            try:
                box = await el.bounding_box()
                # Skip invisible or zero-size elements
                if not box or (box["width"] == 0 and box["height"] == 0):
                    continue

                # Scale up small elements (icons, badges) so they're legible for the vision model
                if box["width"] < MIN_CAPTURE_DIMENSION or box["height"] < MIN_CAPTURE_DIMENSION:
                    scale = max(MIN_CAPTURE_DIMENSION / max(box["width"], 1),
                                MIN_CAPTURE_DIMENSION / max(box["height"], 1))
                    await el.evaluate(
                        "(el, s) => { el.style.transform = `scale(${s})`; el.style.transformOrigin = 'top left'; }",
                        scale,
                    )

                png_bytes = await el.screenshot(type="png")
                src = await el.get_attribute("src") if tag == "img" else None  # SVGs have no src
                heading, paragraph = await _nearest_section_context(el)

                captures.append({
                    "src": src,
                    "tag": tag,
                    "png_bytes": png_bytes,
                    "nearby_heading": heading,
                    "nearby_paragraph": paragraph,
                    "page_url": page_url,
                })
            except Exception:
                continue  # skip elements that error (offscreen, hidden, etc.)

    return captures

async def check_link(client: httpx.AsyncClient, url: str) -> LinkResult:
    try:
        r = await client.head(url, follow_redirects=True, timeout=8)
        return LinkResult(url=url, ok=(r.status_code < 400), status=r.status_code)
    except Exception as e:
        return LinkResult(url=url, ok=False, error=str(e))

async def crawl(root_url: str, max_pages: int = 30, max_depth: int = 2) -> list[PageResult]:
    """BFS crawl from the docs-checker, extended to capture images on each page.
    Each page is loaded once — links checked and images captured in the same visit."""
    bb = Browserbase(api_key=BROWSERBASE_API_KEY)
    session = bb.sessions.create()
    # CDP endpoint lets Playwright drive the cloud browser Browserbase started
    cdp_url = session.connect_url

    base_domain = urlparse(root_url).netloc
    visited: dict[str, PageResult] = {}
    queue = deque([(root_url, 0)])  # each entry is (url, depth)

    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(cdp_url)
        page = await browser.new_page()

        async with httpx.AsyncClient() as http:
            while queue and len(visited) < max_pages:
                url, depth = queue.popleft()
                # Strip fragments so #section anchors don't create duplicate visits
                url = url.split("#")[0]
                if url in visited or depth > max_depth:
                    continue

                clear_output(wait=True)
                print(f"Crawling [{len(visited)+1}/{max_pages}] (depth={depth}): {url[:80]}")

                result = PageResult(url=url)

                try:
                    await page.goto(url, timeout=15000, wait_until="domcontentloaded")

                    # --- Link checking (from docs-checker) ---
                    links = await page.eval_on_selector_all("a[href]", "els => els.map(e => e.href)")
                    for href in links[:30]:
                        if not href.startswith("http"):
                            continue
                        lr = await check_link(http, href)
                        result.links.append(lr)
                        if not lr.ok:
                            result.issues.append(Issue(
                                category="unresolved_reference",
                                description=f"Link returned status {lr.status}",
                                page_url=url,
                                evidence=[href],
                            ))
                        parsed = urlparse(href)
                        if parsed.netloc == base_domain:
                            next_url = href.split("#")[0]
                            if next_url not in visited:
                                queue.append((next_url, depth + 1))

                    # --- Image capture (new) ---
                    result.images = await _capture_page_images(page, url)

                except Exception as e:
                    result.issues.append(Issue(
                        category="unresolved_reference",
                        description="Page failed to load",
                        page_url=url,
                        evidence=[str(e)],
                    ))

                visited[url] = result

        await browser.close()

    clear_output(wait=True)
    total_images = sum(len(r.images) for r in visited.values())
    print(f"✓ Crawled {len(visited)} pages, captured {total_images} images")
    return list(visited.values())

# Step 4: AI Analysis with Cerebras

Next, use Stagehand (for agentic browser automation) to analyze content to identify deeper issues: outdated information, unclear writing, missing context, and grammar errors.

The analysis uses structured outputs for reliable JSON parsing.

In [ ]:
from stagehand import AsyncStagehand

ALLOWED_CATEGORIES = [
    "invalid_snippet", "source_of_truth_mismatch",
    "cross_page_inconsistency", "language_error", "missing_required_element",
]

ISSUES_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "enum": ALLOWED_CATEGORIES},
            "description": {"type": "string"},
            "location": {"type": ["string", "null"]},
            "observed": {"type": ["string", "null"]},
            "expected": {"type": ["string", "null"]},
            "evidence": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["category", "description", "location", "observed", "expected", "evidence"],
        "additionalProperties": False,
    },
}

async def analyze_all(page_results: list[PageResult]) -> list[PageResult]:
    print(f"Analyzing {len(page_results)} pages...")
    sh = AsyncStagehand(
        browserbase_api_key=BROWSERBASE_API_KEY,
        model_api_key=CEREBRAS_API_KEY,
    )
    session = await sh.sessions.start(model_name=STAGEHAND_MODEL)
    try:
        for i, pr in enumerate(page_results):
            print(f"[{i+1}/{len(page_results)}] {pr.url[:60]}...")
            try:
                await session.navigate(url=pr.url)
                # ISSUES_SCHEMA constrains the model's output to recognized issue categories,
                # so the response can be reliably unpacked without additional validation
                resp = await session.extract(
                    instruction=f"Review {pr.url} for issues. Skip broken links.",
                    schema=ISSUES_SCHEMA,
                )
                for item in (resp.data.result if resp and resp.data else []):
                    if isinstance(item, dict):
                        pr.issues.append(Issue(
                            category=item["category"],
                            description=item["description"],
                            page_url=pr.url,
                            location=item.get("location"),
                            observed=item.get("observed"),
                            expected=item.get("expected"),
                            evidence=item.get("evidence", []),
                        ))
            except Exception as e:
                print(f"  Error: {e}")
    finally:
        # Always close the Stagehand session, even if a page errors mid-loop
        await session.end()
    print(f"✓ Analysis complete — {sum(len(p.issues) for p in page_results)} issues found")
    return page_results

# Step 5: Scan Source Files + Generate Alt Text
 
Scan the source repo for every image reference in your MDX files, match each one to its captured screenshot, and caption it with Gemma 4 31B. The model receives the screenshot plus the nearby heading and paragraph, and decides whether each image is decorative or informative in the same call. Decorative images get an explicit `alt=""` rather than a generated description — screen readers skip empty alt attributes rather than narrating something that adds no information.

In [ ]:
MARKDOWN_IMAGE_RE = re.compile(r"!\[([^\]]*)\]\(([^)\s]+)(?:\s+\"[^\"]*\")?\)")
JSX_TAG_RE = re.compile(r"<img\b[^>]*?/?>", re.IGNORECASE)
SRC_ATTR_RE = re.compile(r"src=[\"']([^\"']+)[\"']", re.IGNORECASE)
ALT_ATTR_RE = re.compile(r"alt=[\"']([^\"']*)[\"']", re.IGNORECASE)
 
@dataclass
class ImageRef:
    file_path: str
    src: str
    alt_text: str | None
    line_number: int
    status: str  # "missing" | "weak" | "decorative-likely" | "ok"
 
def classify(alt_text: str | None, src: str) -> str:
    # Paths containing these hints are likely decorative UI chrome —
    # give them a head start so the model can confirm rather than guess
    decorative_hints = {"icon", "logo", "spacer", "divider"}
    if alt_text is None or alt_text.strip() == "":
        return "decorative-likely" if any(h in src.lower() for h in decorative_hints) else "missing"
    if alt_text.strip().lower() in {"image", "screenshot", "photo"}:
        return "weak"
    return "ok"
 
def scan_file(path: Path, docs_root: Path) -> list[ImageRef]:
    text = path.read_text(encoding="utf-8")
    rel_path = str(path.relative_to(docs_root))
    refs = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        # Handle both Markdown image syntax: ![alt](src)
        for m in MARKDOWN_IMAGE_RE.finditer(line):
            alt, src = m.group(1) or None, m.group(2)
            refs.append(ImageRef(rel_path, src, alt, line_no, classify(alt, src)))
        # Handle JSX <img> tags
        for tag_match in JSX_TAG_RE.finditer(line):
            tag = tag_match.group(0)
            src_match = SRC_ATTR_RE.search(tag)
            if not src_match:
                continue
            alt_match = ALT_ATTR_RE.search(tag)
            alt = alt_match.group(1) if alt_match else None
            refs.append(ImageRef(rel_path, src_match.group(1), alt, line_no, classify(alt, src_match.group(1))))
    return refs
 
def scan_docs(docs_root: str) -> list[ImageRef]:
    root = Path(docs_root)
    # Recursively glob all MDX files — change *.mdx to *.md for plain Markdown projects
    return [ref for path in sorted(root.rglob("*.mdx")) for ref in scan_file(path, root)]
 
class AltTextResult(BaseModel):
    is_decorative: bool = Field(description="True if this image is purely decorative")
    alt_text: str = Field(description="Suggested alt text; empty string if decorative")
    confidence: Literal["high", "low"] = "high"
    uncertainty_note: str = ""
 
ALT_TEXT_SYSTEM_PROMPT = """You write alt text for images embedded in technical documentation.
 
First decide: is this image decorative (logo, icon, divider) or does it convey
content a screen reader user needs (diagram, screenshot, chart)?
 
If decorative: set is_decorative=true, alt_text="".
If not: one concise sentence, no "Image of" or "Screenshot of". Describe what is
actually in the image using the provided heading and paragraph as context — don't
just repeat the caption text. If anything is illegible or ambiguous, set
confidence="low" and explain what's uncertain in uncertainty_note."""
 
def get_cerebras_client():
    raw = AsyncOpenAI(api_key=CEREBRAS_API_KEY, base_url="https://api.cerebras.ai/v1")
    return instructor.from_openai(raw)
 
async def caption_one(client, capture: dict, semaphore: asyncio.Semaphore) -> AltTextResult:
    async with semaphore:
        # Encode the raw PNG bytes as base64 for the multimodal API call
        encoded = base64.b64encode(capture["png_bytes"]).decode("ascii")
        # Pass the surrounding section text so the model can describe the image
        # in context rather than in isolation
        context = (
            f"Section heading: {capture['nearby_heading']}\nNearby paragraph: {capture['nearby_paragraph']}"
            if capture["nearby_heading"] or capture["nearby_paragraph"]
            else "(no surrounding text found)"
        )
        return await client.chat.completions.create(
            model=CEREBRAS_MODEL,
            response_model=AltTextResult,  # instructor extracts this typed model from the response
            messages=[
                {"role": "system", "content": ALT_TEXT_SYSTEM_PROMPT},
                {"role": "user", "content": [
                    {"type": "text", "text": f"Page context:\n{context}\n\nGenerate alt text for this image."},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}},
                ]},
            ],
            temperature=0.2,  # low temperature for consistent, factual descriptions
        )
 
async def caption_all(captures: list[dict], concurrency: int = 5) -> list[AltTextResult]:
    # Semaphore caps parallel API calls to avoid hitting rate limits
    semaphore = asyncio.Semaphore(concurrency)
    client = get_cerebras_client()
    results = await asyncio.gather(*[caption_one(client, cap, semaphore) for cap in captures])
    decorative = sum(1 for r in results if r.is_decorative)
    low_conf = sum(1 for r in results if r.confidence == "low")
    print(f"✓ Captioned {len(results)} images — {decorative} decorative, {low_conf} flagged low confidence")
    return results
 
print("✓ Scanner and captioner ready")

# Step 6: Write Back to Source
 
Match each caption to its source file, write the updated MDX files to an output directory, and flag low-confidence suggestions with an inline comment. Review the diff before committing anything.

In [ ]:
def _low_confidence_comment(note: str) -> str:
    return f"{{/* ALT TEXT NEEDS REVIEW: model flagged low confidence — {note or 'see model output'} */}}"
 
def _apply_to_line(line: str, src: str, alt_text: str) -> str:
    def markdown_sub(m):
        return f"![{alt_text}]({m.group(2)})" if m.group(2) == src else m.group(0)
    new_line = MARKDOWN_IMAGE_RE.sub(markdown_sub, line)
    if new_line != line:
        return new_line
    def jsx_sub(m):
        tag = m.group(0)
        src_match = SRC_ATTR_RE.search(tag)
        if not src_match or src_match.group(1) != src:
            return tag
        if ALT_ATTR_RE.search(tag):
            return ALT_ATTR_RE.sub(f'alt="{alt_text}"', tag)
        return tag.replace("<img", f'<img alt="{alt_text}"', 1)
    return JSX_TAG_RE.sub(jsx_sub, line)
 
def write_back(
    caption_pairs: list[tuple[dict, AltTextResult]],
    source_refs: list[ImageRef],
    docs_root: str,
    output_dir: str,
) -> dict:
    docs_root = Path(docs_root)
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
 
    # Build a lookup from src path to ImageRef so we can match
    # browser-captured images back to specific lines in the MDX source
    refs_by_src = {ref.src: ref for ref in source_refs}
    by_file: dict[str, list] = {}
    unmatched = []
 
    for capture, result in caption_pairs:
        src = capture.get("src")
        if src is None:
            # Inline SVGs have no src — can't be safely matched to a source line
            unmatched.append(capture)
            continue
        # Try matching by URL path first (strips query strings), then by full src
        ref = refs_by_src.get(urlparse(src).path) or refs_by_src.get(src)
        if not ref:
            unmatched.append(capture)
            continue
        by_file.setdefault(ref.file_path, []).append((ref, result))
 
    summary = {}
    for rel_path, entries in by_file.items():
        source = docs_root / rel_path
        dest = output_dir / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        lines = source.read_text(encoding="utf-8").splitlines(keepends=True)
        # Group changes by line number so we can process the file in a single pass
        by_line: dict[int, list] = {}
        for ref, result in entries:
            by_line.setdefault(ref.line_number, []).append((ref, result))
        output_lines, changed = [], 0
        for line_no, line in enumerate(lines, start=1):
            line_entries = by_line.get(line_no, [])
            if not line_entries:
                output_lines.append(line)
                continue
            new_line = line
            for ref, result in line_entries:
                updated = _apply_to_line(new_line, ref.src, result.alt_text)
                if updated != new_line:
                    changed += 1
                    new_line = updated
                # Insert a review comment above the line when the model wasn't confident
                if result.confidence == "low":
                    indent = re.match(r"\s*", line).group(0)
                    output_lines.append(f"{indent}{_low_confidence_comment(result.uncertainty_note)}\n")
            output_lines.append(new_line)
        dest.write_text("".join(output_lines), encoding="utf-8")
        summary[rel_path] = changed
 
    if unmatched:
        print(f"⚠ {len(unmatched)} image(s) could not be matched to source (inline SVG or unknown src):")
        for cap in unmatched:
            print(f"  [{cap['tag']}] {cap['page_url']}  src={cap['src']!r}")
 
    return summary

print("✓ Write-back ready")

# Step 7: Run the Pipeline

Configure your target site and local source directory, then run each step in order.

`DOCS_URL` points Browserbase at the live rendered site for crawling and screenshots. `DOCS_SOURCE_DIR` points the scanner at your local MDX files for write-back — both are needed since the browser sees the rendered output and the write-back edits the source.

In [ ]:
DOCS_URL = "https://inference-docs.cerebras.ai"
DOCS_SOURCE_DIR = "/path/to/your/docs/source"  # local path to your MDX source
OUTPUT_DIR = "/tmp/alt-text-review"
MAX_PAGES = 30
MAX_DEPTH = 2

# Crawl and capture images
page_results = await crawl(DOCS_URL, max_pages=MAX_PAGES, max_depth=MAX_DEPTH)

# Analyze pages for quality issues
page_results = await analyze_all(page_results)

# Scan source files and caption all captured images
source_refs = scan_docs(DOCS_SOURCE_DIR)
all_captures = [img for pr in page_results for img in pr.images]
print(f"Found {len(source_refs)} image references in source, {len(all_captures)} images captured from site")

caption_results = await caption_all(all_captures, concurrency=5)
caption_pairs = list(zip(all_captures, caption_results))

# Write back to output directory
summary = write_back(caption_pairs, source_refs, DOCS_SOURCE_DIR, OUTPUT_DIR)
for file_path, count in summary.items():
    print(f"  {file_path}: {count} alt attribute(s) updated")

print(f"\n✓ Review changes with: diff -r {DOCS_SOURCE_DIR} {OUTPUT_DIR}")

# Step 8: Display Results
 
Extend the docs-checker's report to include alt text suggestions alongside quality issues.

In [ ]:
CEREBRAS_DARK = "#1a1a2e"
CEREBRAS_ORANGE = "#f97316"
CEREBRAS_GRAY = "#2d2d44"
CEREBRAS_TEXT = "#e5e5e5"
 
def display_report(page_results: list[PageResult], caption_pairs: list, root_url: str):
    all_issues = [i for pr in page_results for i in pr.issues]
    category_counts = Counter(i.category for i in all_issues)
    decorative = sum(1 for _, r in caption_pairs if r.is_decorative)
    low_conf = sum(1 for _, r in caption_pairs if r.confidence == "low")
    captioned = sum(1 for _, r in caption_pairs if not r.is_decorative)
 
    summary_rows = "".join([
        f'<tr><td style="padding:8px;border-bottom:1px solid {CEREBRAS_GRAY};">{cat}</td>'
        f'<td style="padding:8px;border-bottom:1px solid {CEREBRAS_GRAY};text-align:right;">{cnt}</td></tr>'
        for cat, cnt in category_counts.most_common()
    ])
    alt_rows = "".join([
        f'<tr>'
        f'<td style="padding:8px;border-bottom:1px solid {CEREBRAS_GRAY};font-family:monospace;font-size:0.85em;">{cap.get("src") or "inline SVG"}</td>'
        f'<td style="padding:8px;border-bottom:1px solid {CEREBRAS_GRAY};">{result.alt_text or "<em>decorative</em>"}</td>'
        f'<td style="padding:8px;border-bottom:1px solid {CEREBRAS_GRAY};color:{"#f97316" if result.confidence == "low" else "#888"};">{result.confidence}</td>'
        f'</tr>'
        for cap, result in caption_pairs[:20]
    ])
 
    display(HTML(f'''
    <div style="background:{CEREBRAS_DARK};color:{CEREBRAS_TEXT};padding:30px;border-radius:12px;font-family:system-ui;">
      <h1 style="color:{CEREBRAS_ORANGE};margin-bottom:5px;">Docs Quality + Accessibility Report</h1>
      <p style="color:#888;">Site: <a href="{root_url}" style="color:{CEREBRAS_ORANGE};">{root_url}</a></p>
      <p style="color:#888;">Pages: {len(page_results)} | Quality issues: {len(all_issues)} | Images captioned: {captioned} | Decorative: {decorative} | Low confidence: {low_conf}</p>
      <h2 style="color:{CEREBRAS_TEXT};border-bottom:2px solid {CEREBRAS_ORANGE};padding-bottom:10px;">Quality Issues</h2>
      <table style="width:100%;border-collapse:collapse;margin:15px 0;">
        <tr style="background:{CEREBRAS_GRAY};"><th style="padding:10px;text-align:left;">Category</th><th style="padding:10px;text-align:right;">Count</th></tr>
        {summary_rows}
      </table>
      <h2 style="color:{CEREBRAS_TEXT};border-bottom:2px solid {CEREBRAS_ORANGE};padding-bottom:10px;margin-top:30px;">Alt Text Suggestions</h2>
      <table style="width:100%;border-collapse:collapse;margin:15px 0;">
        <tr style="background:{CEREBRAS_GRAY};"><th style="padding:10px;text-align:left;">Image</th><th style="padding:10px;text-align:left;">Suggested Alt Text</th><th style="padding:10px;text-align:left;">Confidence</th></tr>
        {alt_rows}
      </table>
      <div style="margin-top:30px;padding-top:20px;border-top:1px solid {CEREBRAS_GRAY};text-align:center;color:#666;">Powered by Cerebras + Browserbase + Gemma 4 31B</div>
    </div>'''))
 
display_report(page_results, caption_pairs, DOCS_URL)

## Example Output

Real output from a test run against a fixture page with an architecture diagram and a logo icon:

```diff
- ![](/images/architecture-diagram.svg)
+ ![A line connecting a Client block to an Inference API block.](/images/architecture-diagram.svg)

- <img src="/images/cerebras-logo-icon.svg" />
+ <img alt="" src="/images/cerebras-logo-icon.svg" />
```

The diagram caption describes what's visually present — two labeled boxes and a connecting line. For sparse diagrams, check whether captions draw on the surrounding page context or default to describing only what's visible in the image.

The logo was correctly identified as decorative and given an explicit `alt=""` — the right accessibility pattern, since screen readers skip empty alt attributes rather than narrating something that adds no information.

When the model flags a caption as low-confidence, a review comment is inserted directly above the line in the diff:

```diff
+ {/* ALT TEXT NEEDS REVIEW: model flagged low confidence — diagram has minimal labeling, relationship between components is inferred */}
- ![](/images/architecture-overview.svg)
+ ![Diagram showing the client sending a request to the inference API](/images/architecture-overview.svg)
```

## Limitations

- **Inline SVG with no `src` can't be auto-correlated to source.** These are reported separately rather than written back, since guessing a source location from DOM position risks inserting alt text at the wrong place in the file.
- **Decorative detection is model-judged, not rule-based.** Borderline cases are worth spot-checking before trusting the classification at scale.
- **This generates suggestions, not final copy.** Review the PR diff — a vision model can confidently mis-describe a dense diagram or misread small text in a screenshot.

## Next steps

- Add retry/backoff around the captioning step for rate-limit handling at scale.
- Review the diff in `OUTPUT_DIR` before committing — check any captions flagged `ALT TEXT NEEDS REVIEW` closely before applying them to your source.